In [ ]:
########################################################
# CÁLCULO DE SOBREPOSIÇÃO SEMÂNTICA
#######################################################

import numpy as np
import pandas as pd
from scipy.spatial.distance import pdist, squareform
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer

import nltk

nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords

stopwords_pt = list(stopwords.words('portuguese'))


def auditar_redundancia_maxima_precisao(caminho_planilha):
  print('Carregando os dados...')
  df = pd.read_excel(caminho_planilha)

  coluna_codigo = df.columns[8]  # Coluna I
  coluna_desc = df.columns[14]  # Coluna O

  print(
      f'Mapeamento ativo -> Código: "{coluna_codigo}" (Coluna I) | Descrição:'
      f' "{coluna_desc}" (Coluna O)'
  )

  df = df.dropna(subset=[coluna_codigo, coluna_desc]).copy()

  print('Vetorizando os textos (TF-IDF)...')
  textos = df[coluna_desc].astype(str).tolist()
  vectorizer = TfidfVectorizer(max_features=1500, stop_words=stopwords_pt)
  X_global = vectorizer.fit_transform(textos)

  print('Reduzindo dimensionalidade com alta precisão (10 componentes SVD)...')
  svd = TruncatedSVD(n_components=10, random_state=42)
  X_reduced = svd.fit_transform(X_global)

  for d in range(10):
    df[f'SVD_{d+1}'] = X_reduced[:, d]

  print('Calculando o centro espacial multidimensional de cada código...')
  colunas_svd = [f'SVD_{d+1}' for d in range(10)]
  centros = df.groupby(coluna_codigo)[colunas_svd].mean()
  codigos_lista = centros.index.tolist()
  coords = centros.values

  if len(coords) < 2:
    print('Códigos insuficientes para comparação.')
    return

  print('Executando matriz de distâncias vetorizada...')
  dist_matrix = squareform(pdist(coords, metric='euclidean'))

  relatorio_redundancia = []

  # FILTRO SUPER RIGOROSO: Distância menor e volume maior para eliminar ruídos
  LIMIAR_DISTANCIA = 0.03
  n = len(codigos_lista)

  print('Filtrando apenas duplicações críticas e de alto volume...')
  for i in range(n):
    for j in range(i + 1, n):
      dist = dist_matrix[i, j]
      if dist < LIMIAR_DISTANCIA:
        cod_a = codigos_lista[i]
        cod_b = codigos_lista[j]

        sub_a = df[df[coluna_codigo] == cod_a]
        sub_b = df[df[coluna_codigo] == cod_b]

        # Exige pelo menos 30 documentos em cada código para garantir relevância estrutural
        if len(sub_a) < 30 or len(sub_b) < 30:
          continue

        relatorio_redundancia.append({
            'Codigo_Principal': cod_a,
            'Codigo_Redundante_Sugerido': cod_b,
            'Distancia_Semantica': round(float(dist), 4),
            'Volume_Cod_A': len(sub_a),
            'Volume_Cod_B': len(sub_b),
            'Status': (
                'Eliminar e Absorver (Distinção Administrativa Redundante)'
            ),
        })

  print('Salvando o relatório hiper-filtrado em Excel...')
  df_redundancia = pd.DataFrame(relatorio_redundancia)
  if not df_redundancia.empty:
    df_redundancia = df_redundancia.sort_values(
        by='Distancia_Semantica', ascending=True
    )
    df_redundancia.to_excel(
        'relatorio_codigos_redundantes_top.xlsx', index=False
    )
    print(
        'Sucesso absoluto! Relatório final gerado com'
        f' {len(df_redundancia)} pares de alta prioridade para fusão.'
    )
  else:
    print(
        'Nenhum par atingiu o critério hiper-restrito. Tente ajustar o'
        ' LIMIAR_DISTANCIA para 0.04.'
    )


# Execução com filtro máximo:
auditar_redundancia_maxima_precisao('/content/dados_AP.xlsx')

In [ ]:
#####################################################
# GERA GRÁFICOS DE SOBREPOSIÇÃO
###################################################

import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer

import nltk

nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords

stopwords_pt = list(stopwords.words('portuguese'))


def gerar_graficos_redundancia_top(caminho_planilha):
  print('Carregando os dados e gerando gráficos para os 9 pares...')
  df = pd.read_excel(caminho_planilha)

  coluna_codigo = df.columns[8]  # Coluna I
  coluna_desc = df.columns[14]  # Coluna O

  df = df.dropna(subset=[coluna_codigo, coluna_desc]).copy()

  # Vetorização e SVD para recuperar o espaço 2D ideal para a visualização gráfica
  textos = df[coluna_desc].astype(str).tolist()
  vectorizer = TfidfVectorizer(max_features=1500, stop_words=stopwords_pt)
  X_global = vectorizer.fit_transform(textos)

  svd = TruncatedSVD(n_components=2, random_state=42)
  X_reduced = svd.fit_transform(X_global)

  df['SVD_1'] = X_reduced[:, 0]
  df['SVD_2'] = X_reduced[:, 1]

  # Carrega o relatório gerado com os 9 pares prioritários
  try:
    df_top = pd.read_excel('relatorio_codigos_redundantes_top.xlsx')
  except FileNotFoundError:
    print(
        'O arquivo "relatorio_codigos_redundantes_top.xlsx" não foi encontrado.'
        ' Execute o script anterior primeiro.'
    )
    return

  os.makedirs('graficos_redundancia_top', exist_ok=True)

  # Plota o gráfico individual para cada um dos 9 pares redundantes
  for idx, row in df_top.iterrows():
    cod_a = row['Codigo_Principal']
    cod_b = row['Codigo_Redundante_Sugerido']
    dist = row['Distancia_Semantica']

    sub_a = df[df[coluna_codigo] == cod_a]
    sub_b = df[df[coluna_codigo] == cod_b]

    plt.figure(figsize=(8, 5))
    plt.scatter(
        sub_a['SVD_1'],
        sub_a['SVD_2'],
        color='royalblue',
        label=f'Código {cod_a} (n={len(sub_a)})',
        alpha=0.6,
        s=60,
    )
    plt.scatter(
        sub_b['SVD_1'],
        sub_b['SVD_2'],
        color='darkorange',
        label=f'Código {cod_b} (n={len(sub_b)})',
        alpha=0.6,
        s=60,
    )

    plt.title(
        f'Fusão Recomendada (Distância: {dist})\nCódigos Redundantes:'
        f' {cod_a} vs {cod_b}',
        color='darkblue',
        fontsize=11,
        fontweight='bold',
    )
    plt.xlabel('Componente Principal 1')
    plt.ylabel('Componente Principal 2')
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.6)

    nome_arquivo = f"graficos_redundancia_top/fusao_{str(cod_a).replace('.', '_')}_e_{str(cod_b).replace('.', '_')}.png"
    plt.savefig(nome_arquivo, bbox_inches='tight')
    plt.close()

  print(
      'Pronto! Os 9 gráficos de validação visual foram salvos na pasta'
      ' "graficos_redundancia_top/".'
  )


# Executa a geração dos gráficos:
gerar_graficos_redundancia_top('/content/dados_AP.xlsx')

In [ ]:
#####################################################################
# INSPEÇÃO DE PALAVRAS-CHAVE (TESTE DE ILUSÃO VOCABULAR)
######################################################################

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

import nltk

nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords

stopwords_pt = list(stopwords.words('portuguese'))

# Adiciona algumas palavras burocráticas comuns que costumam poluir descrições administrativas
palavras_extras = [
    'ref',
    'referente',
    'doc',
    'documento',
    'copia',
    'via',
    'anexo',
    'processo',
]
stopwords_pt.extend(palavras_extras)


def inspecionar_palavras_chave_pares(caminho_planilha):
  print('Carregando os dados para inspeção qualitativa...')
  df = pd.read_excel(caminho_planilha)

  coluna_codigo = df.columns[8]  # Coluna I
  coluna_desc = df.columns[14]  # Coluna O

  df = df.dropna(subset=[coluna_codigo, coluna_desc]).copy()

  try:
    df_top = pd.read_excel('relatorio_codigos_redundantes_top.xlsx')
  except FileNotFoundError:
    print(
        'O arquivo "relatorio_codigos_redundantes_top.xlsx" não foi encontrado.'
    )
    return

  print(
      '\n' + '=' * 70
  )
  print('INSPEÇÃO DE PALAVRAS-CHAVE DOS 9 PARES (TESTE DE ILUSÃO VOCABULAR)')
  print('=' * 70)

  for idx, row in df_top.iterrows():
    cod_a = row['Codigo_Principal']
    cod_b = row['Codigo_Redundante_Sugerido']
    dist = row['Distancia_Semantica']

    textos_a = df[df[coluna_codigo] == cod_a][coluna_desc].astype(str).tolist()
    textos_b = df[df[coluna_codigo] == cod_b][coluna_desc].astype(str).tolist()

    # Função rápida para pegar as top 5 palavras mais marcantes de um grupo
    def get_top_keywords(textos):
      vec = TfidfVectorizer(max_features=100, stop_words=stopwords_pt)
      try:
        X = vec.fit_transform(textos)
        somas = X.sum(axis=0).A1
        palavras = vec.get_feature_names_out()
        ranking = sorted(zip(palavras, somas), key=lambda x: x[1], reverse=True)
        return [p[0] for p in ranking[:5]]
      except:
        return []

    keywords_a = get_top_keywords(textos_a)
    keywords_b = get_top_keywords(textos_b)

    print(f'\nPar {idx+1}: Código {cod_a}  vs  Código {cod_b} (Distância: {dist})')
    print(f'   -> Termos-chave no {cod_a}: {keywords_a}')
    print(f'   -> Termos-chave no {cod_b}: {keywords_b}')
    print('-' * 70)


# Executa a inspeção:
inspecionar_palavras_chave_pares('/content/dados_AP.xlsx')